# 06 - Geração de PDFs de Prescrição (Nutricional & Suplementação)
Projeto: **Performance Híbrida / Hyrox Analytics** — Unity Catalog

Este notebook gera automaticamente um PDF de prescrição por sessão presente
na tabela `gold_prescricao_diaria`, salvando os rascunhos em
`/Volumes/workspace/default/treinos_trainingpeaks/prescricoes_pendentes/`
para revisão do profissional antes do envio ao atleta.

In [0]:
%pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CÉLULA 12: Gerar um único PDF com todos os treinos e orientações do atleta

from pyspark.sql import functions as F

# Importar bibliotecas para criação do PDF
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Spacer,
    Paragraph,
    Table,
    TableStyle,
    PageBreak
)

# Importar bibliotecas nativas para trabalhar com arquivos e textos
from datetime import datetime
from html import escape
import os
import re

# Definir a pasta criada anteriormente para PDFs que aguardam revisão
caminho_pendentes = (
    "/Volumes/workspace/default/treinos_trainingpeaks/"
    "prescricoes_pendentes"
)

# Garantir que a pasta existe antes de criar o arquivo
os.makedirs(caminho_pendentes, exist_ok=True)

# Ler a tabela Gold com todas as sessões prescritas
df_gold_pdf = (
    spark.table("workspace.default.gold_prescricao_diaria")
    .select(
        "id_treino",
        "id_atleta",
        "nome_completo",
        "peso_kg",
        "data_treino",
        "modalidade",
        "titulo_treino",
        "duracao_minutos",
        "distancia_km",
        "zona_fc_predominante",
        "classificacao_intensidade",
        "prescricao_carboidrato_intra",
        "prescricao_cafeina_pre_treino",
        "prescricao_sodio_eletrolitos",
        "prescricao_tamponamento_bicarbonato",
        "protocolo_cronico_diario"
    )
    .orderBy("data_treino", "titulo_treino")
)

# Trazer os registros para a geração de um único documento
registros = [
    linha.asDict()
    for linha in df_gold_pdf.collect()
]

# Impedir a criação de arquivo vazio
if len(registros) == 0:
    raise Exception(
        "A gold_prescricao_diaria não possui sessões para gerar o PDF."
    )

# Verificar se há somente um atleta no documento semanal
atletas_documento = sorted(
    list(
        {
            str(registro["nome_completo"]).strip()
            for registro in registros
            if registro["nome_completo"] is not None
        }
    )
)

if len(atletas_documento) != 1:
    raise Exception(
        "O PDF semanal deve conter somente um atleta. "
        f"Foram encontrados: {atletas_documento}"
    )

# Função para não mostrar valores nulos ou vazios no PDF
def texto_seguro(valor, padrao="Não informado"):
    if valor is None:
        return padrao

    texto = str(valor).strip()

    if texto == "":
        return padrao

    return texto

# Função para permitir texto seguro dentro dos elementos do ReportLab
def texto_pdf(valor, padrao="Não informado"):
    return escape(texto_seguro(valor, padrao)).replace("\n", "<​br/>")

# Função para criar um nome de arquivo seguro
def limpar_nome_arquivo(texto):
    return re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        str(texto)
    )[:80]

# Identificar dados gerais do documento
nome_atleta = atletas_documento[0]
peso_atleta = texto_seguro(registros[0]["peso_kg"])
primeira_data = texto_seguro(registros[0]["data_treino"])
ultima_data = texto_seguro(registros[-1]["data_treino"])

# Criar nome e caminho do PDF único
nome_arquivo_pdf = (
    f"RASCUNHO_PLANO_TREINOS_"
    f"{limpar_nome_arquivo(nome_atleta)}_"
    f"{limpar_nome_arquivo(primeira_data)}_"
    f"a_{limpar_nome_arquivo(ultima_data)}.pdf"
)

caminho_pdf = os.path.join(
    caminho_pendentes,
    nome_arquivo_pdf
)

# Criar estilos do documento
estilos = getSampleStyleSheet()

estilo_titulo = ParagraphStyle(
    "TituloPlano",
    parent=estilos["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    textColor=colors.HexColor("#163A5F"),
    spaceAfter=7
)

estilo_subtitulo = ParagraphStyle(
    "SubtituloPlano",
    parent=estilos["Normal"],
    alignment=TA_CENTER,
    fontSize=9,
    leading=12,
    textColor=colors.HexColor("#555555"),
    spaceAfter=14
)

estilo_secao = ParagraphStyle(
    "SecaoPlano",
    parent=estilos["Heading2"],
    fontSize=12,
    leading=15,
    textColor=colors.HexColor("#163A5F"),
    spaceBefore=11,
    spaceAfter=7
)

estilo_treino = ParagraphStyle(
    "TituloTreino",
    parent=estilos["Heading3"],
    fontSize=11,
    leading=14,
    textColor=colors.HexColor("#163A5F"),
    spaceBefore=10,
    spaceAfter=5
)

estilo_texto = ParagraphStyle(
    "TextoPlano",
    parent=estilos["BodyText"],
    fontSize=9,
    leading=13,
    alignment=TA_LEFT
)

estilo_nota = ParagraphStyle(
    "NotaPlano",
    parent=estilos["BodyText"],
    fontSize=8,
    leading=11,
    textColor=colors.HexColor("#555555"),
    alignment=TA_LEFT
)

# Configurar o documento A4
documento = SimpleDocTemplate(
    caminho_pdf,
    pagesize=A4,
    rightMargin=1.5 * cm,
    leftMargin=1.5 * cm,
    topMargin=1.4 * cm,
    bottomMargin=1.4 * cm
)

elementos = []

# Cabeçalho do documento
elementos.append(
    Paragraph(
        "PLANO DE SUPLEMENTAÇÃO E HIDRATAÇÃO PARA TREINAMENTOS",
        estilo_titulo
    )
)

elementos.append(
    Paragraph(
        "RASCUNHO PARA REVISÃO PROFISSIONAL — "
        "não enviar ao atleta antes da validação final",
        estilo_subtitulo
    )
)

# Resumo geral do atleta e período
elementos.append(
    Paragraph(
        "1. Identificação do plano",
        estilo_secao
    )
)

dados_gerais = [
    ["Atleta", texto_pdf(nome_atleta)],
    ["Peso registrado", texto_pdf(f"{peso_atleta} kg")],
    ["Período dos treinos", texto_pdf(f"{primeira_data} a {ultima_data}")],
    ["Quantidade de sessões", str(len(registros))],
    [
        "Data de geração",
        datetime.now().strftime("%d/%m/%Y %H:%M")
    ]
]

tabela_geral = Table(
    [
        [
            Paragraph(f"<b>{chave}</b>", estilo_texto),
            Paragraph(valor, estilo_texto)
        ]
        for chave, valor in dados_gerais
    ],
    colWidths=[5.0 * cm, 11.5 * cm]
)

tabela_geral.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EAF1F7")),
        ("TEXTCOLOR", (0, 0), (0, -1), colors.HexColor("#163A5F")),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C9D7E3")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 7),
        ("RIGHTPADDING", (0, 0), (-1, -1), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6)
    ])
)

elementos.append(tabela_geral)

# Criar uma seção de orientação para cada treino do período

# CÉLULA 12: Gerar um único PDF com todos os treinos e orientações do atleta

from pyspark.sql import functions as F

# Importar bibliotecas para criação do PDF
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Spacer,
    Paragraph,
    Table,
    TableStyle,
    PageBreak
)

# Importar bibliotecas nativas para trabalhar com arquivos e textos
from datetime import datetime
from html import escape
import os
import re

# Definir a pasta criada anteriormente para PDFs que aguardam revisão
caminho_pendentes = (
    "/Volumes/workspace/default/treinos_trainingpeaks/"
    "prescricoes_pendentes"
)

# Garantir que a pasta existe antes de criar o arquivo
os.makedirs(caminho_pendentes, exist_ok=True)

# Ler a tabela Gold com todas as sessões prescritas
df_gold_pdf = (
    spark.table("workspace.default.gold_prescricao_diaria")
    .select(
        "id_treino",
        "id_atleta",
        "nome_completo",
        "peso_kg",
        "data_treino",
        "modalidade",
        "titulo_treino",
        "duracao_minutos",
        "distancia_km",
        "zona_fc_predominante",
        "classificacao_intensidade",
        "prescricao_carboidrato_intra",
        "prescricao_cafeina_pre_treino",
        "prescricao_sodio_eletrolitos",
        "prescricao_tamponamento_bicarbonato",
        "protocolo_cronico_diario"
    )
    .orderBy("data_treino", "titulo_treino")
)

# Trazer os registros para a geração de um único documento
registros = [
    linha.asDict()
    for linha in df_gold_pdf.collect()
]

# Impedir a criação de arquivo vazio
if len(registros) == 0:
    raise Exception(
        "A gold_prescricao_diaria não possui sessões para gerar o PDF."
    )

# Verificar se há somente um atleta no documento semanal
atletas_documento = sorted(
    list(
        {
            str(registro["nome_completo"]).strip()
            for registro in registros
            if registro["nome_completo"] is not None
        }
    )
)

if len(atletas_documento) != 1:
    raise Exception(
        "O PDF semanal deve conter somente um atleta. "
        f"Foram encontrados: {atletas_documento}"
    )

# Função para não mostrar valores nulos ou vazios no PDF
def texto_seguro(valor, padrao="Não informado"):
    if valor is None:
        return padrao

    texto = str(valor).strip()

    if texto == "":
        return padrao

    return texto

# Função para permitir texto seguro dentro dos elementos do ReportLab
def texto_pdf(valor, padrao="Não informado"):
    return escape(texto_seguro(valor, padrao)).replace("\n", "<​br/>")

# Função para criar um nome de arquivo seguro
def limpar_nome_arquivo(texto):
    return re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        str(texto)
    )[:80]

# Identificar dados gerais do documento
nome_atleta = atletas_documento[0]
peso_atleta = texto_seguro(registros[0]["peso_kg"])
primeira_data = texto_seguro(registros[0]["data_treino"])
ultima_data = texto_seguro(registros[-1]["data_treino"])

# Criar nome e caminho do PDF único
nome_arquivo_pdf = (
    f"RASCUNHO_PLANO_TREINOS_"
    f"{limpar_nome_arquivo(nome_atleta)}_"
    f"{limpar_nome_arquivo(primeira_data)}_"
    f"a_{limpar_nome_arquivo(ultima_data)}.pdf"
)

caminho_pdf = os.path.join(
    caminho_pendentes,
    nome_arquivo_pdf
)

# Criar estilos do documento
estilos = getSampleStyleSheet()

estilo_titulo = ParagraphStyle(
    "TituloPlano",
    parent=estilos["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    textColor=colors.HexColor("#163A5F"),
    spaceAfter=7
)

estilo_subtitulo = ParagraphStyle(
    "SubtituloPlano",
    parent=estilos["Normal"],
    alignment=TA_CENTER,
    fontSize=9,
    leading=12,
    textColor=colors.HexColor("#555555"),
    spaceAfter=14
)

estilo_secao = ParagraphStyle(
    "SecaoPlano",
    parent=estilos["Heading2"],
    fontSize=12,
    leading=15,
    textColor=colors.HexColor("#163A5F"),
    spaceBefore=11,
    spaceAfter=7
)

estilo_treino = ParagraphStyle(
    "TituloTreino",
    parent=estilos["Heading3"],
    fontSize=11,
    leading=14,
    textColor=colors.HexColor("#163A5F"),
    spaceBefore=10,
    spaceAfter=5
)

estilo_texto = ParagraphStyle(
    "TextoPlano",
    parent=estilos["BodyText"],
    fontSize=9,
    leading=13,
    alignment=TA_LEFT
)

estilo_nota = ParagraphStyle(
    "NotaPlano",
    parent=estilos["BodyText"],
    fontSize=8,
    leading=11,
    textColor=colors.HexColor("#555555"),
    alignment=TA_LEFT
)

# Configurar o documento A4
documento = SimpleDocTemplate(
    caminho_pdf,
    pagesize=A4,
    rightMargin=1.5 * cm,
    leftMargin=1.5 * cm,
    topMargin=1.4 * cm,
    bottomMargin=1.4 * cm
)

elementos = []

# Cabeçalho do documento
elementos.append(
    Paragraph(
        "PLANO DE SUPLEMENTAÇÃO E HIDRATAÇÃO PARA TREINAMENTOS",
        estilo_titulo
    )
)

elementos.append(
    Paragraph(
        "RASCUNHO PARA REVISÃO PROFISSIONAL — "
        "não enviar ao atleta antes da validação final",
        estilo_subtitulo
    )
)

# Resumo geral do atleta e período
elementos.append(
    Paragraph(
        "1. Identificação do plano",
        estilo_secao
    )
)

dados_gerais = [
    ["Atleta", texto_pdf(nome_atleta)],
    ["Peso registrado", texto_pdf(f"{peso_atleta} kg")],
    ["Período dos treinos", texto_pdf(f"{primeira_data} a {ultima_data}")],
    ["Quantidade de sessões", str(len(registros))],
    [
        "Data de geração",
        datetime.now().strftime("%d/%m/%Y %H:%M")
    ]
]

tabela_geral = Table(
    [
        [
            Paragraph(f"<b>{chave}</b>", estilo_texto),
            Paragraph(valor, estilo_texto)
        ]
        for chave, valor in dados_gerais
    ],
    colWidths=[5.0 * cm, 11.5 * cm]
)

tabela_geral.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EAF1F7")),
        ("TEXTCOLOR", (0, 0), (0, -1), colors.HexColor("#163A5F")),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C9D7E3")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 7),
        ("RIGHTPADDING", (0, 0), (-1, -1), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6)
    ])
)

elementos.append(tabela_geral)

# Criar uma seção de orientação para cada treino do período
elementos.append(
    Paragraph(
        "2. Orientações por treinamento",
        estilo_secao
    )
)

for indice, treino in enumerate(registros, start=1):

    # Exibir título e identificação da sessão
    titulo_secao = (
        f"{indice}. {texto_pdf(treino['data_treino'])} — "
        f"{texto_pdf(treino['titulo_treino'])}"
    )

    elementos.append(
        Paragraph(
            titulo_secao,
            estilo_treino
        )
    )

    # Organizar as características do treino
    dados_sessao = [
        [
            Paragraph("<b>Modalidade</b>", estilo_texto),
            Paragraph(texto_pdf(treino["modalidade"]), estilo_texto)
        ],
        [
            Paragraph("<b>Duração</b>", estilo_texto),
            Paragraph(
                texto_pdf(f"{texto_seguro(treino['duracao_minutos'])} min"),
                estilo_texto
            )
        ],
        [
            Paragraph("<b>Distância</b>", estilo_texto),
            Paragraph(
                texto_pdf(f"{texto_seguro(treino['distancia_km'])} km"),
                estilo_texto
            )
        ],
        [
            Paragraph("<b>Zona predominante</b>", estilo_texto),
            Paragraph(texto_pdf(treino["zona_fc_predominante"]), estilo_texto)
        ],
        [
            Paragraph("<b>Intensidade</b>", estilo_texto),
            Paragraph(
                texto_pdf(treino["classificacao_intensidade"]),
                estilo_texto
            )
        ]
    ]

    tabela_sessao = Table(
        dados_sessao,
        colWidths=[4.2 * cm, 12.3 * cm]
    )

    tabela_sessao.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#F3F7FB")),
            ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C9D7E3")),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 7),
            ("RIGHTPADDING", (0, 0), (-1, -1), 7),
            ("TOPPADDING", (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5)
        ])
    )

    elementos.append(tabela_sessao)
    elementos.append(Spacer(1, 7))

    # Organizar as recomendações agudas da sessão
    dados_orientacao = [
        [
            Paragraph("<b>Carboidrato durante o treino</b>", estilo_texto),
            Paragraph(
                texto_pdf(treino["prescricao_carboidrato_intra"]),
                estilo_texto
            )
        ],
        [
            Paragraph("<b>Cafeína pré-treino</b>", estilo_texto),
            Paragraph(
                texto_pdf(treino["prescricao_cafeina_pre_treino"]),
                estilo_texto
            )
        ],
        [
            Paragraph("<b>Sódio e eletrólitos</b>", estilo_texto),
            Paragraph(
                texto_pdf(treino["prescricao_sodio_eletrolitos"]),
                estilo_texto
            )
        ],
        [
            Paragraph("<b>Tamponamento / bicarbonato</b>", estilo_texto),
            Paragraph(
                texto_pdf(
                    treino["prescricao_tamponamento_bicarbonato"]
                ),
                estilo_texto
            )
        ]
    ]

    tabela_orientacao = Table(
        dados_orientacao,
        colWidths=[5.2 * cm, 11.3 * cm]
    )

    tabela_orientacao.setStyle(
        TableStyle([
            ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EAF1F7")),
            ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C9D7E3")),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 7),
            ("RIGHTPADDING", (0, 0), (-1, -1), 7),
            ("TOPPADDING", (0, 0), (-1, -1), 7),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 7)
        ])
    )

    elementos.append(tabela_orientacao)
    elementos.append(Spacer(1, 9))

# Exibir o protocolo crônico uma única vez, pois é diário
elementos.append(
    Paragraph(
        "3. Protocolo diário de base",
        estilo_secao
    )
)

elementos.append(
    Paragraph(
        texto_pdf(registros[0]["protocolo_cronico_diario"]),
        estilo_texto
    )
)

elementos.append(Spacer(1, 12))

# Incluir campo de revisão profissional sem expor seção de risco clínico
elementos.append(
    Paragraph(
        "4. Revisão profissional",
        estilo_secao
    )
)

elementos.append(
    Paragraph(
        "Revisado por: ______________________________________________",
        estilo_texto
    )
)

elementos.append(Spacer(1, 9))

elementos.append(
    Paragraph(
        "Data da revisão: ____ / ____ / ______"
        "    Status: ☐ Aprovado   ☐ Ajustar   ☐ Não liberar",
        estilo_texto
    )
)

elementos.append(Spacer(1, 12))

# Inserir aviso geral de responsabilidade profissional
elementos.append(
    Paragraph(
        "Este plano apresenta orientações parametrizadas de suplementação "
        "e hidratação a partir das características dos treinamentos e de "
        "critérios conservadores definidos no sistema. A aplicação final "
        "deve ser revisada e individualizada por profissional habilitado, "
        "considerando tolerância, estratégia nutricional global e contexto "
        "específico do atleta.",
        estilo_nota
    )
)

# Construir e salvar o PDF no Volume
documento.build(elementos)

# Confirmar que o arquivo foi criado no caminho definido
if not os.path.exists(caminho_pdf):
    raise Exception("O PDF não foi encontrado após a geração.")

print("PDF único gerado com sucesso!")
print("Quantidade de sessões incluídas:", len(registros))
print("Nome do arquivo:", nome_arquivo_pdf)
print("Local para revisão:", caminho_pdf)

PDF único gerado com sucesso!
Quantidade de sessões incluídas: 4
Nome do arquivo: RASCUNHO_PLANO_TREINOS_Rafael_Souto_de_Souza_2026-09-07_a_2026-09-08.pdf
Local para revisão: /Volumes/workspace/default/treinos_trainingpeaks/prescricoes_pendentes/RASCUNHO_PLANO_TREINOS_Rafael_Souto_de_Souza_2026-09-07_a_2026-09-08.pdf
